# 03 — Forecasting and development-only tuning

Compare all 30,490 series, screen candidate settings, and select statistical orders for fixed demand groups using development windows only.

## Imports

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, json, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'data/calendar.csv').exists())
DATA = ROOT / 'data'
OUT = DATA / 'processed'
RESULTS = ROOT / 'reports/results'
FIGURES = ROOT / 'reports/figures'
for folder in [OUT, RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.venv/matplotlib'))


'/Users/moha/Machine-Learning-Project/Project 1/.venv/matplotlib'

In [3]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':100, 'axes.spines.top':False, 'axes.spines.right':False})
COLORS = ['#2563eb', '#059669', '#ea580c', '#7c3aed']


## Load prepared arrays

In [4]:
sales = np.load(OUT / 'sales.npy', mmap_mode='r')
prices = np.load(OUT / 'prices.npy', mmap_mode='r')
metadata = pd.read_csv(OUT / 'metadata.csv')
calendar = pd.read_csv(OUT / 'calendar.csv', parse_dates=['date'])
config = json.loads((ROOT / 'configs/comparison.json').read_text())
data = SimpleNamespace(root=ROOT)


## ARIMA, SARIMA, and LightGBM

In [5]:
import joblib
import lightgbm as lgb
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
panel=pd.read_csv(OUT / 'comparison_series.csv')
panel_rows=panel.row.to_numpy()
assert len(panel)==len(metadata)==30490 and panel.series_id.tolist()==metadata.series_id.iloc[panel_rows].tolist()
display(pd.Series(config).to_frame('setting'))


,setting
horizon,28
development_cutoffs,"[1857, 1885]"
test_cutoff,1913
history_days,728
training_stride,13
seed,42
n_estimators,200
num_leaves,31
learning_rate,0.05
baseline_floor,0.05


## Calendar features

In [6]:
CATEGORICAL = ['item_id_code','store_id_code','dept_id_code','cat_id_code',
               'state_id_code','weekday','month','event_1','event_2']


In [7]:
def calendar_features(rows, days):
    cal = calendar.iloc[days-1]
    meta = metadata.iloc[rows]
    f = meta[CATEGORICAL[:5]].reset_index(drop=True).astype('int16')
    f['weekday'] = cal.wday.to_numpy(dtype='int8')
    f['month'] = cal.month.to_numpy(dtype='int8')
    f['event_1'] = cal.event_name_1_code.to_numpy(dtype='int16')
    f['event_2'] = cal.event_name_2_code.to_numpy(dtype='int16')
    f['trend'] = (days-1).astype('float32')
    angle = 2*np.pi*cal.date.dt.day_of_year.to_numpy()/365.25
    f['annual_sin'], f['annual_cos'] = np.sin(angle).astype('float32'), np.cos(angle).astype('float32')
    f['snap'] = np.select([meta.state_id.to_numpy()==s for s in ['CA','TX','WI']],
                          [cal['snap_'+s].to_numpy() for s in ['CA','TX','WI']]).astype('int8')
    return f


## Lag and rolling features

In [8]:
def demand_features(rows, days):
    t = days-1
    f = {f'lag_{lag}':sales[rows,t-lag].astype('float32') for lag in [28,35,42,56]}
    window = sales[rows[:,None],t[:,None]-28-np.arange(28)]
    f['mean_7'] = window[:,:7].mean(axis=1,dtype='float32')
    f['mean_28'] = window.mean(axis=1,dtype='float32')
    f['std_28'] = window.std(axis=1,dtype='float32')
    f['zero_rate_28'] = (window==0).mean(axis=1).astype('float32')
    return pd.DataFrame(f)


## Features available at the forecast origin

In [9]:
def origin_features(rows, days, origins):
    rows,days,origins = map(np.asarray,(rows,days,origins))
    assert ((days-origins>=1)&(days-origins<=28)).all() and origins.min()>=56
    f = pd.concat([calendar_features(rows,days),demand_features(rows,days)],axis=1)
    recent = sales[rows[:,None],origins[:,None]-1-np.arange(28)]
    last_weekday = origins-1-((origins-days)%7)
    weekly = sales[rows[:,None],last_weekday[:,None]-7*np.arange(8)]
    f['horizon'] = (days-origins).astype('int8')
    f['origin_mean_7'] = recent[:,:7].mean(axis=1,dtype='float32')
    f['origin_mean_28'] = recent.mean(axis=1,dtype='float32')
    f['origin_zero_rate_28'] = (recent==0).mean(axis=1).astype('float32')
    f['weekday_baseline'] = weekly.mean(axis=1,dtype='float32')
    return f


## Training dates and forecast indices

In [10]:
def training_indices(cutoff):
    rows,days = [],[]
    start = cutoff-config['history_days']+1
    for offset in range(config['training_stride']):
        group = np.arange(offset,len(metadata),config['training_stride'])
        targets = np.arange(start+offset,cutoff+1,config['training_stride'])
        rows.append(np.repeat(group,len(targets)))
        days.append(np.tile(targets,len(group)))
    return np.concatenate(rows),np.concatenate(days)


In [11]:
def forecast_indices(panel_rows, cutoff):
    rows = np.repeat(np.asarray(panel_rows),config['horizon'])
    days = np.tile(np.arange(cutoff+1,cutoff+config['horizon']+1),len(panel_rows))
    return rows,days,np.full(len(rows),cutoff)


## Comparable error metrics

In [12]:
def rmsse_scales(history):
    scales=[]
    for values in history:
        active=np.flatnonzero(values)
        differences=np.diff(values[active[0]:].astype('float64')) if len(active) else []
        scales.append(float(np.mean(np.square(differences))) if len(differences) else 0.)
    return np.asarray(scales)


In [13]:
def score_forecast(prediction, actual, scales):
    assert prediction.shape==actual.shape and np.isfinite(prediction).all()
    assert (prediction>=0).all()
    error=prediction.astype('float64')-actual
    valid=scales>0
    rmsse=np.sqrt(np.mean(error[valid]**2,axis=1)/scales[valid])
    return dict(mae=float(np.abs(error).mean()),rmse=float(np.sqrt(np.mean(error**2))),
                mean_rmsse=float(rmsse.mean()),rmsse_series=int(valid.sum()),
                all_series=len(actual))


## Check RMSSE against hand calculations

In [14]:
toy=np.array([[0,0,2,4,3],[1,2,3,4,5]],dtype='float32')
np.testing.assert_allclose(rmsse_scales(toy),[2.5,1.0])
actual=np.ones((2,28))
assert score_forecast(actual,actual,np.ones(2))['mean_rmsse']==0
assert score_forecast(actual+1,actual,np.ones(2))['mean_rmsse']==1


## LightGBM settings

In [15]:
model_parameters=dict(objective='poisson',random_state=config['seed'],verbosity=-1,
    n_estimators=config['n_estimators'],learning_rate=config['learning_rate'],
    num_leaves=config['num_leaves'],max_bin=63,min_child_samples=100,
    reg_lambda=1.0,colsample_bytree=0.9,n_jobs=1)
MODEL_DIR=ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)


## Train the pooled LightGBM correction

In [16]:
def fit_lightgbm(cutoff):
    started=time.perf_counter()
    rows,days=training_indices(cutoff)
    assert days.max()<=cutoff and len(np.unique(rows))==len(metadata)
    horizons=(rows*17+days*13)%28+1
    X=origin_features(rows,days,days-horizons)
    offsets=np.maximum(X.weekday_baseline.to_numpy(),config['baseline_floor'])
    y=sales[rows,days-1].astype('float32')/offsets
    fitted=lgb.LGBMRegressor(**model_parameters)
    fitted.fit(X,y,categorical_feature=CATEGORICAL,sample_weight=offsets)
    joblib.dump(fitted,MODEL_DIR / f'comparison_lightgbm_d{cutoff}.joblib')
    return fitted,time.perf_counter()-started


## Predict the full panel with LightGBM

In [17]:
def predict_lightgbm(fitted, cutoff):
    prediction=np.empty((len(panel),config['horizon']),dtype='float32')
    for start in range(0,len(panel),3000):
        rows,days,origins=forecast_indices(panel_rows[start:start+3000],cutoff)
        X=origin_features(rows,days,origins)
        offsets=np.maximum(X.weekday_baseline.to_numpy(),config['baseline_floor'])
        values=np.maximum(fitted.predict(X)*offsets,0)
        prediction[start:start+3000]=values.reshape(-1,config['horizon'])
    return prediction


## Statistical models and CPU worker controls

In [18]:
from joblib import Parallel, delayed, parallel_config
from threadpoolctl import threadpool_limits
import hashlib
PREDICTIONS=DATA / 'interim'
PREDICTIONS.mkdir(exist_ok=True)


In [19]:
def fit_optimizer(history,kind,maxiter,diffuse=False,solver='lbfgs'):
    if kind=='ARIMA':
        model=ARIMA(history,order=tuple(config['arima_order']),trend='n')
    else:
        model=SARIMAX(history,order=tuple(config['arima_order']),
                      seasonal_order=tuple(config['seasonal_order']),trend='n')
    if diffuse:
        model.ssm.initialize_approximate_diffuse()
    if kind=='ARIMA':
        return model.fit(method_kwargs={'maxiter':maxiter,'method':solver})
    return model.fit(maxiter=maxiter,disp=False,method=solver)


In [20]:
def valid_fitted_forecast(fitted):
    values=np.asarray(fitted.forecast(config['horizon']),dtype='float32')
    return np.isfinite(values).all()


In [21]:
def fit_with_initialization_retry(history,kind):
    attempts=[(False,'lbfgs','fitted'),(True,'lbfgs','diffuse_retry'),
              (True,'powell','optimizer_retry')]
    for diffuse,solver,status in attempts:
        try:
            fitted=fit_optimizer(history,kind,config['maxiter'],diffuse,solver)
            if not fitted.mle_retvals.get('converged',True):
                fitted=fit_optimizer(history,kind,config['maxiter']*2,diffuse,solver)
            if not valid_fitted_forecast(fitted):
                raise FloatingPointError('Non-finite model or forecast')
            return fitted,status
        except (np.linalg.LinAlgError,ValueError,FloatingPointError) as error:
            last_error=error
    raise RuntimeError('All numerical retries failed') from last_error


## Fit a series and record optimizer diagnostics

In [22]:
def fit_statistical(history,kind):
    if np.ptp(history)==0:
        return np.full(config['horizon'],history[-1]),dict(
            status='constant',converged=False,warnings=0,negative_raw_forecasts=0)
    with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=1):
        warnings.simplefilter('always')
        fitted,status=fit_with_initialization_retry(history,kind)
    raw=np.asarray(fitted.forecast(config['horizon']))
    assert np.isfinite(raw).all()
    info=dict(status=status,converged=bool(fitted.mle_retvals.get('converged',True)),
              warnings=len(captured),negative_raw_forecasts=int((raw<0).sum()))
    return np.maximum(raw,0),info


## Identify the exact configuration and input data

In [23]:
import ast, importlib.metadata
input_info={p.name:dict(bytes=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns)
            for p in sorted(DATA.glob('*.csv'))}
source=json.loads((ROOT / 'notebooks/modeling/03_model_comparison.ipynb').read_text())
code_text='\n'.join(''.join(x['source']) for x in source['cells'] if x['cell_type']=='code')
critical=['fit_optimizer','fit_with_initialization_retry','valid_fitted_forecast',
          'fit_statistical','fit_row','fit_batch']
estimator_code='|'.join(ast.dump(node,include_attributes=False)
    for node in ast.parse(code_text).body if isinstance(node,ast.FunctionDef) and node.name in critical)
identity=dict(config=config,inputs=input_info,
    code_hash=hashlib.sha256(code_text.encode()).hexdigest(),
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','statsmodels','lightgbm']})


In [24]:
cache_config={key:config[key] for key in ['arima_order','seasonal_order',
    'history_days','maxiter','horizon','batch_size','run_version']}
cache_identity=dict(config=cache_config,inputs=input_info,versions=identity['versions'],
                    estimator_hash=hashlib.sha256(estimator_code.encode()).hexdigest())
run_id=hashlib.sha256(json.dumps(cache_identity,sort_keys=True).encode()).hexdigest()[:16]
CHECKPOINTS=PREDICTIONS / ('full_panel_checkpoints_'+run_id)
CHECKPOINTS.mkdir(exist_ok=True)
print('Checkpoint run:',run_id)


Checkpoint run: a7d982227033ecec


## Write completed batches atomically

In [25]:
def checkpoint_name(kind,cutoff,start,stop):
    return CHECKPOINTS / f'{kind}_d{cutoff}_{start:05d}_{stop:05d}.npz'

def save_batch(path,forecast,details):
    frame=pd.DataFrame(details)
    arrays={column:frame[column].to_numpy() for column in frame}
    for column in frame:
        if frame[column].dtype.kind in 'OUS':
            arrays[column]=frame[column].to_numpy(dtype=str)
    temporary=path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary,forecast=forecast,**arrays)
    os.replace(temporary,path)


## Process one batch of product/store series

In [26]:
def fit_row(history,kind,row,cutoff):
    try:
        return fit_statistical(history,kind)
    except Exception as error:
        raise RuntimeError(f'{kind}, d_{cutoff}, series row {row}: {error}') from error


In [27]:
def fit_batch(kind,cutoff,start,stop):
    path=checkpoint_name(kind,cutoff,start,stop)
    if path.exists():
        return str(path)
    local_sales=np.load(OUT / 'sales.npy',mmap_mode='r')
    forecast=np.empty((stop-start,config['horizon']),dtype='float32')
    details=[]
    for position,row in enumerate(range(start,stop)):
        history=local_sales[row,cutoff-config['history_days']:cutoff].astype('float64')
        values,info=fit_row(history,kind,row,cutoff)
        forecast[position]=values
        details.append(dict(row=row,model=kind,cutoff=cutoff,**info))
    save_batch(path,forecast,details)
    return str(path)


## Restore forecasts in their original row order

In [28]:
def collect_batches(kind,cutoff,ranges):
    forecast=np.empty((len(panel),config['horizon']),dtype='float32')
    records=[]
    for start,stop in ranges:
        with np.load(checkpoint_name(kind,cutoff,start,stop),allow_pickle=False) as batch:
            assert batch['row'].tolist()==list(range(start,stop))
            values=batch['forecast']
            assert values.shape==(stop-start,config['horizon']) and np.isfinite(values).all()
            forecast[start:stop]=values
            records.append(pd.DataFrame({k:batch[k] for k in batch.files if k!='forecast'}))
    return forecast,pd.concat(records,ignore_index=True).to_dict('records')


## Run all series with parallel CPU workers

In [29]:
def predict_statistical(cutoff,kind):
    ranges=[(start,min(start+config['batch_size'],len(panel)))
            for start in range(0,len(panel),config['batch_size'])]
    pending=[(start,stop) for start,stop in ranges
             if not checkpoint_name(kind,cutoff,start,stop).exists()]
    done=len(ranges)-len(pending)
    print(f'{kind} d_{cutoff}: {done}/{len(ranges)} batches cached',flush=True)
    with parallel_config(backend='loky',inner_max_num_threads=1):
        stream=Parallel(n_jobs=config['workers'],return_as='generator_unordered')(
            delayed(fit_batch)(kind,cutoff,start,stop) for start,stop in pending)
        for _ in stream:
            done+=1
            print(f'{kind} d_{cutoff}: {done}/{len(ranges)} batches',flush=True)
    return collect_batches(kind,cutoff,ranges)


## Save comparable scores and predictions

In [30]:
records,fit_diagnostics=[],[]
cutoffs=[*config['development_cutoffs'],config['test_cutoff']]
PREDICTIONS=DATA / 'interim'
PREDICTIONS.mkdir(exist_ok=True)

def record_result(name, cutoff, prediction, seconds, cached_batches=0):
    actual=sales[panel_rows,cutoff:cutoff+config['horizon']]
    scales=rmsse_scales(sales[panel_rows,:cutoff])
    phase='local_test' if cutoff==config['test_cutoff'] else 'development'
    records.append(dict(model=name,cutoff=cutoff,phase=phase,wall_seconds_this_run=seconds,cached_batches=cached_batches,
                        **score_forecast(prediction,actual,scales)))
    np.save(PREDICTIONS / f'full_panel_{name}_d{cutoff}.npy',prediction)
    pd.DataFrame(records).to_csv(RESULTS / 'full_panel_scores.csv',index=False)


## Run the two development windows and the previous local test

In [31]:
def run_original_models():
    for cutoff in cutoffs:
        fitted,elapsed=fit_lightgbm(cutoff)
        record_result('LightGBM',cutoff,predict_lightgbm(fitted,cutoff),elapsed)
        for name in ['ARIMA','SARIMA']:
            cached=sum(checkpoint_name(name,cutoff,start,min(start+config['batch_size'],len(panel))).exists()
                       for start in range(0,len(panel),config['batch_size']))
            started=time.perf_counter()
            prediction,diagnostics=predict_statistical(cutoff,name)
            fit_diagnostics.extend(diagnostics)
            record_result(name,cutoff,prediction,time.perf_counter()-started,cached_batches=cached)
            pd.DataFrame(fit_diagnostics).to_csv(PREDICTIONS / 'full_panel_fit_diagnostics.csv',index=False)
        print(pd.DataFrame(records).tail(3).to_string(index=False),flush=True)


In [32]:
ml_names=['calendar_features','demand_features','origin_features','training_indices',
          'forecast_indices','fit_lightgbm','predict_lightgbm']
parsed=ast.parse(code_text)
ml_nodes=[ast.dump(node,include_attributes=False) for node in parsed.body
          if isinstance(node,ast.FunctionDef) and node.name in ml_names]
ml_nodes += [ast.dump(node,include_attributes=False) for node in parsed.body
    if isinstance(node,ast.Assign) and any(isinstance(target,ast.Name) and target.id=='model_parameters'
                                        for target in node.targets)]
ml_hash=hashlib.sha256('|'.join(ml_nodes).encode()).hexdigest()
expected_ml_hash=json.loads((ROOT / 'configs/tuning.json').read_text())['original_lightgbm_hash']
prior_manifest=json.loads((RESULTS / 'full_panel_manifest.json').read_text())
reuse_original=(prior_manifest['inputs']==input_info and prior_manifest['config']==config
                and prior_manifest['versions']==identity['versions']
                and prior_manifest['run_id']==run_id and ml_hash==expected_ml_hash)


In [33]:
if reuse_original:
    records=pd.read_csv(RESULTS / 'full_panel_scores.csv').to_dict('records')
    fit_diagnostics=pd.read_csv(PREDICTIONS / 'full_panel_fit_diagnostics.csv').to_dict('records')
    assert len(records)==9 and len(fit_diagnostics)==182940
    print('Reusing original results: data, settings, estimator code, and versions match.')
else:
    run_original_models()


Reusing original results: data, settings, estimator code, and versions match.


## Check full-panel identity and prediction integrity

In [34]:
scores=pd.DataFrame(records)
assert len(scores)==9 and set(scores.model)=={'ARIMA','SARIMA','LightGBM'}
for cutoff in cutoffs:
    for name in ['ARIMA','SARIMA','LightGBM']:
        values=np.load(PREDICTIONS / f'full_panel_{name}_d{cutoff}.npy')
        assert values.shape==(len(metadata),28) and np.isfinite(values).all() and (values>=0).all()
display(scores)


,model,cutoff,phase,wall_seconds_this_run,cached_batches,mae,rmse,mean_rmsse,rmsse_series,all_series
0,LightGBM,1857,development,36.610515,0,1.038714,2.252882,0.737348,30490,30490
1,ARIMA,1857,development,0.543200,120,1.040654,2.309305,0.733311,30490,30490
2,SARIMA,1857,development,0.305596,120,1.037606,2.298532,0.732940,30490,30490
3,LightGBM,1885,development,38.267795,0,1.025961,2.106944,0.738100,30490,30490
4,ARIMA,1885,development,73.583857,107,1.046050,2.267792,0.738679,30490,30490
5,SARIMA,1885,development,1212.686070,0,1.042466,2.245494,0.738354,30490,30490
6,LightGBM,1913,local_test,41.297212,0,1.047335,2.153174,0.759525,30490,30490
7,ARIMA,1913,local_test,561.726518,0,1.056983,2.218067,0.761730,30490,30490
8,SARIMA,1913,local_test,1188.120165,0,1.053837,2.203009,0.761337,30490,30490


## Development ranking

In [35]:
ranking=scores[scores.phase=='development'].groupby('model').agg(
    mean_rmsse=('mean_rmsse','mean'),mae=('mae','mean'),rmse=('rmse','mean'))
ranking=ranking.sort_values('mean_rmsse')
display(ranking)
print('Development ranking only; the previously inspected local test is a retrospective check.')


,mean_rmsse,mae,rmse
model,,,
SARIMA,0.735647,1.040036,2.272013
ARIMA,0.735995,1.043352,2.288549
LightGBM,0.737724,1.032338,2.179913


Development ranking only; the previously inspected local test is a retrospective check.


## Record settings and package versions

In [36]:
import platform
manifest=dict(identity,run_id=run_id,series_count=len(panel),python=platform.python_version(),
    series_ids_hash=hashlib.sha256('|'.join(panel.series_id).encode()).hexdigest())
provenance=CHECKPOINTS / 'compatible_cache.json'
manifest['checkpoint_provenance']=json.loads(provenance.read_text()) if provenance.exists() else {}
(RESULTS / 'full_panel_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')


7046

## Development-only tuning and a weekly baseline

Screen candidate orders and training lengths, then confirm each family’s best alternative on all series. The local test never selects settings.

In [37]:
tuning=json.loads((ROOT / 'configs/tuning.json').read_text())
TUNED=PREDICTIONS / 'tuned_models'
TUNED.mkdir(exist_ok=True)
specifications={entry['id']:entry for entry in tuning['candidates']}
original_scores=pd.read_csv(RESULTS / 'full_panel_scores.csv')
original_diagnostics=pd.read_csv(PREDICTIONS / 'full_panel_fit_diagnostics.csv')
original_manifest=json.loads((RESULTS / 'full_panel_manifest.json').read_text())
assert original_manifest['inputs']==input_info
assert original_manifest['config']==config
assert config['test_cutoff'] not in config['development_cutoffs']


## Select screening strata before the earliest development window

In [38]:
selection_origin=min(config['development_cutoffs'])
recent=sales[panel_rows,selection_origin-tuning['selection_history_days']:selection_origin]
strata=panel[['row','series_id','store_id','cat_id']].copy()
strata['selling_rate']=(recent>0).mean(axis=1)
strata['recent_mean']=recent.mean(axis=1)
strata['demand_group']=np.where(strata.selling_rate>=tuning['regular_sales_rate'],
                               'regular','intermittent')
stratum_keys=['store_id','cat_id','demand_group']
strata['volume_band']=strata.groupby(stratum_keys).recent_mean.transform(
    lambda values:pd.qcut(values.rank(method='first'),3,labels=False))
stratum_keys.append('volume_band')
strata['population']=strata.groupby(stratum_keys).row.transform('size')


In [39]:
shuffled=strata.sample(frac=1,random_state=tuning['seed'])
screen=shuffled.groupby(stratum_keys,sort=False).head(tuning['sample_per_stratum'])
screen=screen.sort_values('row').copy()
screen['sample_count']=screen.groupby(stratum_keys).row.transform('size')
screen['weight']=screen.population/screen.sample_count
screen_rows=screen.row.to_numpy()
assert screen.row.is_unique and np.isclose(screen.weight.sum(),len(panel))
screen.to_csv(OUT / 'tuning_screen.csv',index=False)
print(f'Screen {len(screen)} series; full confirmation uses {len(panel):,}.')
display(screen.groupby(['cat_id','demand_group']).size().unstack())


Screen 536 series; full confirmation uses 30,490.


demand_group,intermittent,regular
cat_id,,
FOODS,90,90
HOBBIES,90,86
HOUSEHOLD,90,90


## Fit one statistical candidate

Alternative SARIMA candidates use partially conditional estimation after differencing; forecasts are restored to original sales units.
This reduces CPU cost; original settings remain an unchanged benchmark.

In [40]:
def restore_sales_forecast(history,forecast,specification):
    d=specification['order'][1]
    _,D,_,period=specification['seasonal_order']
    assert d in [0,1] and D in [0,1]
    polynomial=np.array([1.,-1.]) if d else np.array([1.])
    if D:
        seasonal=np.zeros(period+1);seasonal[0],seasonal[-1]=1.,-1.
        polynomial=np.convolve(polynomial,seasonal)
    restored=list(np.asarray(history,dtype='float64'))
    for value in forecast:
        adjustment=sum(polynomial[lag]*restored[-lag] for lag in range(1,len(polynomial)))
        restored.append(float(value)-adjustment)
    return np.asarray(restored[-len(forecast):])


In [41]:
time_index=np.arange(118)
known=10+2*time_index+np.array([0,3,1,5,2,4,1])[time_index%7]
for d in [0,1]:
    test_specification=dict(order=[0,d,1],seasonal_order=[0,1,1,7])
    differenced_forecast=np.full(28,0. if d else 14.)
    restored=restore_sales_forecast(known[:90],differenced_forecast,test_specification)
    np.testing.assert_allclose(restored,known[90:],rtol=0,atol=1e-10)
print('Seasonal and combined differencing reconstruct a known trend and weekly cycle.')


Seasonal and combined differencing reconstruct a known trend and weekly cycle.


In [42]:
def fit_tuning_optimizer(history,specification,diffuse,solver,maxiter):
    arguments=dict(order=tuple(specification['order']),trend=specification['trend'])
    if specification['family']=='ARIMA':
        fitted_model=ARIMA(history,**arguments)
    else:
        fitted_model=SARIMAX(history,seasonal_order=tuple(specification['seasonal_order']),
            simple_differencing=specification.get('simple_differencing',False),**arguments)
    if diffuse:
        fitted_model.ssm.initialize_approximate_diffuse()
    if specification['family']=='ARIMA':
        return fitted_model.fit(method_kwargs={'maxiter':maxiter,'method':solver})
    return fitted_model.fit(maxiter=maxiter,disp=False,method=solver,
                            cov_type='none',low_memory=True)


In [43]:
def tuning_forecast_in_sales_units(fitted,history,specification):
    forecast=np.asarray(fitted.forecast(config['horizon']))
    if specification.get('simple_differencing',False):
        forecast=restore_sales_forecast(history,forecast,specification)
    return forecast.astype('float32')


In [44]:
def fitted_tuning_forecast(history,specification):
    attempts=[(False,'lbfgs','fitted'),(True,'lbfgs','diffuse_retry'),(True,'powell','optimizer_retry')]
    for diffuse,solver,status in attempts:
        try:
            fitted=fit_tuning_optimizer(history,specification,diffuse,solver,tuning['maxiter'])
            if not fitted.mle_retvals.get('converged',True):
                fitted=fit_tuning_optimizer(history,specification,diffuse,solver,2*tuning['maxiter'])
            values=tuning_forecast_in_sales_units(fitted,history,specification)
            if not np.isfinite(values).all():
                raise FloatingPointError('Non-finite forecast')
            return fitted,values,status
        except (np.linalg.LinAlgError,ValueError,FloatingPointError) as error:
            last_error=error
    raise RuntimeError(f"Candidate {specification['id']} failed") from last_error


In [45]:
def fit_tuning_series(history,specification):
    if np.ptp(history)==0:
        return np.full(config['horizon'],history[-1]),dict(
            status='constant',converged=False,warnings=0,negative_raw_forecasts=0)
    with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=1):
        warnings.simplefilter('always')
        fitted,values,status=fitted_tuning_forecast(history,specification)
    details=dict(status=status,converged=bool(fitted.mle_retvals.get('converged',True)),
                 warnings=len(captured),negative_raw_forecasts=int((values<0).sum()))
    return np.maximum(values,0),details


## Fingerprint tuning code and cache each batch

In [46]:
tuning_nodes=['fit_tuning_optimizer','fitted_tuning_forecast','fit_tuning_series','tuning_batch',
              'restore_sales_forecast','tuning_forecast_in_sales_units']
tuning_source=json.loads((ROOT / 'notebooks/modeling/03_model_comparison.ipynb').read_text())
tuning_code='\n'.join(''.join(c['source']) for c in tuning_source['cells'] if c['cell_type']=='code')
tuning_ast='|'.join(ast.dump(node,include_attributes=False) for node in ast.parse(tuning_code).body
    if isinstance(node,ast.FunctionDef) and node.name in tuning_nodes)
tuning_identity=dict(inputs=input_info,versions=identity['versions'],settings=tuning,
    estimator_hash=hashlib.sha256(tuning_ast.encode()).hexdigest(),
    screen_hash=hashlib.sha256(screen.to_csv(index=False).encode()).hexdigest())
tuning_run=hashlib.sha256(json.dumps(tuning_identity,sort_keys=True).encode()).hexdigest()[:16]
TUNING_CACHE=PREDICTIONS / ('tuning_checkpoints_'+tuning_run)
TUNING_CACHE.mkdir(exist_ok=True)
print('Tuning cache:',tuning_run)


Tuning cache: b20cd6a56f55679c


In [47]:
TUNED=TUNED / tuning_run
TUNED.mkdir(exist_ok=True)


In [48]:
def tuning_path(specification,cutoff,rows):
    row_hash=hashlib.sha256(np.asarray(rows,dtype='int64').tobytes()).hexdigest()[:12]
    return TUNING_CACHE / f"{specification['id']}_d{cutoff}_{row_hash}.npz"


In [49]:
def tuning_batch(specification,cutoff,rows):
    path=tuning_path(specification,cutoff,rows)
    if path.exists():
        return str(path)
    local_sales=np.load(OUT / 'sales.npy',mmap_mode='r')
    forecast=np.empty((len(rows),config['horizon']),dtype='float32')
    details=[]
    for position,row in enumerate(rows):
        history=local_sales[row,cutoff-specification['history_days']:cutoff].astype('float64')
        values,info=fit_tuning_series(history,specification)
        forecast[position]=values
        details.append(dict(row=row,model=specification['family'],cutoff=cutoff,**info))
    save_batch(path,forecast,details)
    return str(path)


In [50]:
def collect_tuning_batches(specification,cutoff,batches):
    predictions,frames=[],[]
    for rows in batches:
        with np.load(tuning_path(specification,cutoff,rows),allow_pickle=False) as batch:
            assert batch['row'].tolist()==rows.tolist()
            assert batch['forecast'].shape==(len(rows),config['horizon'])
            assert np.isfinite(batch['forecast']).all() and (batch['forecast']>=0).all()
            predictions.append(batch['forecast'])
            frames.append(pd.DataFrame({k:batch[k] for k in batch.files if k!='forecast'}))
    return np.concatenate(predictions),pd.concat(frames,ignore_index=True)


In [51]:
def predict_tuning_candidate(specification,cutoff,rows,batch_size):
    if specification['id'].endswith('_original'):
        prediction=np.load(PREDICTIONS / f"full_panel_{specification['family']}_d{cutoff}.npy")
        details=original_diagnostics[original_diagnostics.model.eq(specification['family']) & original_diagnostics.cutoff.eq(cutoff)]
        return prediction[rows],details.set_index('row').loc[rows].reset_index()
    batches=[rows[start:start+batch_size] for start in range(0,len(rows),batch_size)]
    pending=[batch for batch in batches if not tuning_path(specification,cutoff,batch).exists()]
    completed=len(batches)-len(pending)
    with parallel_config(backend='loky',inner_max_num_threads=1):
        stream=Parallel(n_jobs=tuning['workers'],return_as='generator_unordered')(
            delayed(tuning_batch)(specification,cutoff,batch) for batch in pending)
        for _ in stream:
            completed+=1
            print(f"{specification['id']} d{cutoff}: {completed}/{len(batches)} batches",flush=True)
    return collect_tuning_batches(specification,cutoff,batches)


## Weight screening errors back to the full series population

In [52]:
def weighted_screen_score(prediction,actual,scales,weights):
    error=prediction.astype('float64')-actual.astype('float64')
    assert error.shape==actual.shape and np.isfinite(error).all()
    valid=scales>0
    per_series_mse=np.mean(error**2,axis=1)
    rmsse=np.sqrt(per_series_mse[valid]/scales[valid])
    return dict(mean_rmsse=float(np.average(rmsse,weights=weights[valid])),
        mae=float(np.average(np.abs(error).mean(axis=1),weights=weights)),
        rmse=float(np.sqrt(np.average(per_series_mse,weights=weights))))

screen_actual={c:sales[screen_rows,c:c+28].astype('float64') for c in config['development_cutoffs']}
screen_scales={c:rmsse_scales(sales[screen_rows,:c]) for c in config['development_cutoffs']}
screen_records=[]


## Screen the candidate grid on development windows only

In [53]:
for specification in tuning['candidates']:
    for cutoff in config['development_cutoffs']:
        prediction,details=predict_tuning_candidate(specification,cutoff,screen_rows,
                                                    tuning['screen_batch_size'])
        metrics=weighted_screen_score(prediction,screen_actual[cutoff],screen_scales[cutoff],
                                      screen.weight.to_numpy())
        screen_records.append(dict(candidate=specification['id'],family=specification['family'],
            cutoff=cutoff,series=len(screen_rows),converged=int(details.converged.sum()),**metrics))
        pd.DataFrame(screen_records).to_csv(RESULTS / 'tuning_screen_scores.csv',index=False)
screen_ranking=pd.DataFrame(screen_records).groupby(['family','candidate']).mean_rmsse.mean()
display(screen_ranking.sort_values())


family  candidate           
ARIMA   arima_original          0.704048
SARIMA  sarima_original         0.704520
ARIMA   arima_011_728           0.704590
SARIMA  sarima_011_0117_728     0.705100
ARIMA   arima_011_364           0.705252
        arima_101c_728          0.715060
SARIMA  sarima_001_0117_728     0.717316
        sarima_001_0117_364     0.718021
ARIMA   arima_101c_364          0.718777
SARIMA  sarima_101c_1007_364    0.718877
Name: mean_rmsse, dtype: float64

## Confirm the strongest alternative for each family on all series

In [54]:
screen_table=screen_ranking.rename('mean_rmsse').reset_index()
alternatives=screen_table[~screen_table.candidate.str.endswith('_original')]
shortlist=alternatives.loc[alternatives.groupby('family').mean_rmsse.idxmin()]
shortlist.to_csv(RESULTS / 'tuning_shortlist.csv',index=False)
confirmation=[]
for candidate in shortlist.candidate:
    specification=specifications[candidate]
    for cutoff in config['development_cutoffs']:
        values,details=predict_tuning_candidate(specification,cutoff,panel_rows,tuning['batch_size'])
        np.save(TUNED / f'{candidate}_d{cutoff}.npy',values)
        details.to_csv(TUNED / f'{candidate}_d{cutoff}_diagnostics.csv',index=False)
        metrics=score_forecast(values,sales[panel_rows,cutoff:cutoff+28],
                               rmsse_scales(sales[panel_rows,:cutoff]))
        confirmation.append(dict(candidate=candidate,family=specification['family'],cutoff=cutoff,**metrics))
        pd.DataFrame(confirmation).to_csv(RESULTS / 'tuning_confirmation_scores.csv',index=False)


sarima_011_0117_728 d1857: 39/120 batches


sarima_011_0117_728 d1857: 40/120 batches


sarima_011_0117_728 d1857: 41/120 batches


sarima_011_0117_728 d1857: 42/120 batches


sarima_011_0117_728 d1857: 43/120 batches


...

sarima_011_0117_728 d1885: 116/120 batches


sarima_011_0117_728 d1885: 117/120 batches


sarima_011_0117_728 d1885: 118/120 batches


sarima_011_0117_728 d1885: 119/120 batches


sarima_011_0117_728 d1885: 120/120 batches


In [55]:
for family in ['ARIMA','SARIMA']:
    for result in original_scores.query('phase=="development" and model==@family').itertuples():
        confirmation.append(dict(candidate=family.lower()+'_original',family=family,
            cutoff=result.cutoff,mean_rmsse=result.mean_rmsse,mae=result.mae,rmse=result.rmse,
            rmsse_series=result.rmsse_series,all_series=result.all_series))
confirmation_scores=pd.DataFrame(confirmation)
confirmation_scores.to_csv(RESULTS / 'tuning_confirmation_scores.csv',index=False)
confirmation_ranking=confirmation_scores.groupby(['family','candidate']).mean_rmsse.mean()


## Select settings for regular and intermittent demand

Groups are fixed using 90 days before the earliest development origin. Choose original or confirmed alternative within each group using development RMSSE only.

In [56]:
def full_confirmation_forecast(candidate,cutoff):
    specification=specifications[candidate]
    if candidate.endswith('_original'):
        return np.load(PREDICTIONS / f"full_panel_{specification['family']}_d{cutoff}.npy")
    return np.load(TUNED / f'{candidate}_d{cutoff}.npy')

group_records=[]
for result in confirmation_scores.itertuples():
    values=full_confirmation_forecast(result.candidate,result.cutoff)
    for group in ['intermittent','regular']:
        rows=strata.loc[strata.demand_group.eq(group),'row'].to_numpy()
        metrics=score_forecast(values[rows],sales[rows,result.cutoff:result.cutoff+28],
                               rmsse_scales(sales[rows,:result.cutoff]))
        group_records.append(dict(family=result.family,candidate=result.candidate,
                                   demand_group=group,cutoff=result.cutoff,**metrics))


In [57]:
group_scores=pd.DataFrame(group_records)
group_scores.to_csv(RESULTS / 'tuning_group_scores.csv',index=False)
group_ranking=group_scores.groupby(['family','demand_group','candidate']).mean_rmsse.mean()
selection=group_ranking.rename('development_rmsse').reset_index()
selection=selection.loc[selection.groupby(['family','demand_group']).development_rmsse.idxmin()]
selected={family:{row.demand_group:specifications[row.candidate]
    for row in selection[selection.family.eq(family)].itertuples()} for family in ['ARIMA','SARIMA']}
series_settings=strata[['row','series_id','demand_group']].copy()
for family in ['ARIMA','SARIMA']:
    series_settings[family]=series_settings.demand_group.map(lambda group:selected[family][group]['id'])
series_settings.to_csv(OUT / 'tuned_series_settings.csv',index=False)
display(selection)


,family,demand_group,candidate,development_rmsse
1,ARIMA,intermittent,arima_original,0.720693
3,ARIMA,regular,arima_original,0.853124
5,SARIMA,intermittent,sarima_original,0.720716
6,SARIMA,regular,sarima_011_0117_728,0.847209


## Freeze selected settings before the retrospective local test

In [58]:
selection_manifest=dict(identity=tuning_identity,run_id=tuning_run,
    selection_cutoffs=config['development_cutoffs'],selection_metric='mean series RMSSE within each fixed demand group',
    screening_series=len(screen_rows),confirmation_series=len(panel),selected=selected,
    selection_scores=selection.to_dict('records'),test_used_for_selection=False)
cache_provenance=TUNING_CACHE / 'compatible_cache.json'
selection_manifest['cache_provenance']=json.loads(cache_provenance.read_text()) if cache_provenance.exists() else {}
selection_manifest['group_counts']={str(group):int(count) for group,count in strata.demand_group.value_counts().items()}
(RESULTS / 'tuning_selection.json').write_text(json.dumps(selection_manifest,indent=2)+'\n')
assert set(selection_manifest['selection_cutoffs'])==set(config['development_cutoffs'])
assert config['test_cutoff'] not in selection_manifest['selection_cutoffs']
print('Selected settings frozen. The local test remains a retrospective assessment.')


Selected settings frozen. The local test remains a retrospective assessment.


## Forecast selected statistical models and retain unchanged LightGBM

In [59]:
def candidate_group_forecast(specification,cutoff,rows):
    path=TUNED / f"{specification['id']}_d{cutoff}.npy"
    if path.exists():
        details=pd.read_csv(path.with_name(path.stem+'_diagnostics.csv')).set_index('row')
        return np.load(path)[rows],details.loc[rows].reset_index()
    return predict_tuning_candidate(specification,cutoff,rows,tuning['batch_size'])


In [60]:
def selected_statistical_forecast(family,cutoff):
    original=specifications[family.lower()+'_original']
    prediction,details=predict_tuning_candidate(original,cutoff,panel_rows,tuning['batch_size'])
    details=details.set_index('row')
    for group,specification in selected[family].items():
        if specification['id'].endswith('_original'):
            continue
        rows=strata.loc[strata.demand_group.eq(group),'row'].to_numpy()
        values,group_details=candidate_group_forecast(specification,cutoff,rows)
        prediction[rows]=values
        details.loc[rows,:]=group_details.set_index('row').loc[rows,details.columns]
    return prediction,details.reset_index()


In [61]:
def seasonal_naive_forecast(cutoff):
    history=sales[panel_rows,cutoff-7:cutoff]
    assert history.shape==(len(panel),7)
    return np.tile(history,(1,4)).astype('float32')

tuned_records,tuned_details=[],[]


In [62]:
def save_tuned_result(name,cutoff,values):
    scales=rmsse_scales(sales[panel_rows,:cutoff])
    metrics=score_forecast(values,sales[panel_rows,cutoff:cutoff+28],scales)
    phase='local_test' if cutoff==config['test_cutoff'] else 'development'
    tuned_records.append(dict(model=name,cutoff=cutoff,phase=phase,**metrics))
    np.save(PREDICTIONS / f'tuned_{name}_d{cutoff}.npy',values)
    pd.DataFrame(tuned_records).to_csv(RESULTS / 'tuned_scores.csv',index=False)

for cutoff in [*config['development_cutoffs'],config['test_cutoff']]:
    save_tuned_result('SeasonalNaive',cutoff,seasonal_naive_forecast(cutoff))
    lightgbm=np.load(PREDICTIONS / f'full_panel_LightGBM_d{cutoff}.npy')
    save_tuned_result('LightGBM',cutoff,lightgbm)


In [63]:
for cutoff in [*config['development_cutoffs'],config['test_cutoff']]:
    for family in ['ARIMA','SARIMA']:
        values,details=selected_statistical_forecast(family,cutoff)
        save_tuned_result(family,cutoff,values)
        tuned_details.append(details)
        pd.concat(tuned_details,ignore_index=True).to_csv(
            PREDICTIONS / 'tuned_fit_diagnostics.csv',index=False)
    print(pd.DataFrame(tuned_records).query('cutoff==@cutoff').to_string(index=False),flush=True)


        model  cutoff       phase      mae     rmse  mean_rmsse  rmsse_series  all_series
SeasonalNaive    1857 development 1.210618 2.760262    0.958631         30490       30490
     LightGBM    1857 development 1.038714 2.252882    0.737348         30490       30490
        ARIMA    1857 development 1.040654 2.309305    0.733311         30490       30490
       SARIMA    1857 development 1.034863 2.296853    0.732234         30490       30490


        model  cutoff       phase      mae     rmse  mean_rmsse  rmsse_series  all_series
SeasonalNaive    1885 development 1.232355 2.759459    0.975596         30490       30490
     LightGBM    1885 development 1.025961 2.106944    0.738100         30490       30490
        ARIMA    1885 development 1.046050 2.267792    0.738679         30490       30490
       SARIMA    1885 development 1.045539 2.292407    0.738430         30490       30490


sarima_011_0117_728 d1913: 1/14 batches


sarima_011_0117_728 d1913: 2/14 batches


sarima_011_0117_728 d1913: 3/14 batches


...

sarima_011_0117_728 d1913: 11/14 batches


sarima_011_0117_728 d1913: 12/14 batches


sarima_011_0117_728 d1913: 13/14 batches


sarima_011_0117_728 d1913: 14/14 batches


        model  cutoff      phase      mae     rmse  mean_rmsse  rmsse_series  all_series
SeasonalNaive    1913 local_test 1.243972 2.676895    0.997242         30490       30490
     LightGBM    1913 local_test 1.047335 2.153174    0.759525         30490       30490
        ARIMA    1913 local_test 1.056983 2.218067    0.761730         30490       30490
       SARIMA    1913 local_test 1.049055 2.199251    0.760355         30490       30490


## Check the baseline, selected forecasts, and full-panel coverage

In [64]:
tuned_scores=pd.DataFrame(tuned_records)
assert len(tuned_scores)==12 and set(tuned_scores.all_series)=={30490}
for cutoff in [*config['development_cutoffs'],config['test_cutoff']]:
    baseline=np.load(PREDICTIONS / f'tuned_SeasonalNaive_d{cutoff}.npy')
    np.testing.assert_array_equal(baseline[:,:7],sales[panel_rows,cutoff-7:cutoff])
    np.testing.assert_array_equal(baseline[:,7:14],baseline[:,:7])
    for name in ['ARIMA','SARIMA','LightGBM','SeasonalNaive']:
        values=np.load(PREDICTIONS / f'tuned_{name}_d{cutoff}.npy')
        assert values.shape==(30490,28) and np.isfinite(values).all() and (values>=0).all()
for family in ['ARIMA','SARIMA']:
    new_mean=tuned_scores.query('phase=="development" and model==@family').mean_rmsse.mean()
    old_mean=original_scores.query('phase=="development" and model==@family').mean_rmsse.mean()
    assert new_mean<=old_mean+1e-10


In [65]:
manifest=dict(selection_manifest,series_count=len(panel),config=config,
    series_ids_hash=original_manifest['series_ids_hash'],python=platform.python_version(),
    notebook_code_hash=hashlib.sha256(tuning_code.encode()).hexdigest())
(RESULTS / 'tuned_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')


8028

Development selection kept the original ARIMA settings and changed SARIMA only for regular sellers.
Local-test SARIMA mean RMSSE improved from 0.7613 to 0.7604; LightGBM remains lowest at 0.7595.